# CropCop Track A — Post-Training Preflight

This notebook performs only pre-metric recovery and artifact-availability checks. It does not run validation replay, robustness, XAI, selection, V1-test inference, Track-B predictions, or Track-C candidate evaluation.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

repo = Path(os.environ["CROPCOP_REPO_ROOT"]).resolve()
analysis_sha = os.environ["CROPCOP_ANALYSIS_SHA"].strip()
account_id = os.environ.get("CROPCOP_ACCOUNT_ID", "").strip()
if len(analysis_sha) != 40:
    raise RuntimeError("CROPCOP_ANALYSIS_SHA must be a full 40-character SHA")
observed = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if observed != analysis_sha:
    raise RuntimeError(f"exact analysis checkout mismatch: expected={analysis_sha}, observed={observed}")
os.environ.setdefault("CROPCOP_NOTEBOOK_STARTED_MONOTONIC", repr(time.monotonic()))
os.environ.setdefault("CROPCOP_NOTEBOOK_HARD_LIMIT_SECONDS", "43200")
os.environ.setdefault("CROPCOP_NOTEBOOK_FINALIZATION_MARGIN_SECONDS", "3600")
print({"analysis_sha": analysis_sha, "account_id": account_id or None, "repo": str(repo)})


In [ ]:
required = ["CROPCOP_ACCOUNT_ID", "CROPCOP_PREFLIGHT_SPEC", "CROPCOP_PREFLIGHT_OUTPUT"]
missing = [name for name in required if not os.environ.get(name, "").strip()]
if missing:
    raise RuntimeError(f"missing required environment variables: {missing}")
if account_id not in {"K1", "K2", "K3"}:
    raise RuntimeError("CROPCOP_ACCOUNT_ID must be K1, K2, or K3")
subprocess.run([
    sys.executable,
    str(repo / "journal_extension/scripts/prepare_tracka_v12_posttraining_account.py"),
    "--repo-root", str(repo),
    "--spec", os.environ["CROPCOP_PREFLIGHT_SPEC"],
    "--analysis-source-git-commit", analysis_sha,
    "--output-dir", os.environ["CROPCOP_PREFLIGHT_OUTPUT"],
], cwd=repo, check=True)


In [ ]:
manifest = Path(os.environ["CROPCOP_PREFLIGHT_OUTPUT"]) / f"{account_id}_POSTTRAINING_PREFLIGHT_MANIFEST.json"
payload = json.loads(manifest.read_text(encoding="utf-8"))
if payload.get("status") != "PASS":
    raise RuntimeError("preflight manifest is not PASS")
print(json.dumps({
    "status": payload["status"],
    "account_id": payload["account_id"],
    "continuation_recovery_count": payload["continuation_recovery_count"],
    "historical_availability_report_sha256": payload["historical_availability_report_sha256"],
    "manifest_sha256": payload["manifest_sha256"],
}, indent=2))
